# 06 · Reproducir el champion desde el Model Registry

Notebook limpio: no depende de ninguna variable de los anteriores. Carga el modelo registrado como
`models:/favorita-perecederos-modelo@champion`, reconstruye el conjunto de prueba con el mismo código y la misma
muestra (huella SHA-256 verificada) y comprueba que el RMSLE y el costo de negocio de la prueba (Co:Cu 1:3) son
**exactamente** los registrados en su corrida final. La salida del champion es el pedido sugerido: el costo se calcula
directo sobre ella.

In [1]:
import os
import sys

sys.path.insert(0, '..')
os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
os.environ.setdefault('MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR', 'false')
os.environ.setdefault('TQDM_DISABLE', '1')
os.environ['MLFLOW_ALLOW_MODEL_CODE_EXECUTION'] = 'true'    # modelo del servidor de tracking del proyecto (formato skops)

import mlflow
import numpy as np
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient

from src.datos import cargar_muestra
from src.pipeline import SERIE, construir_features, dividir, rmsle

load_dotenv('../.env')
URI = 'models:/favorita-perecederos-modelo@champion'
cliente = MlflowClient()
version = cliente.get_model_version_by_alias('favorita-perecederos-modelo', 'champion')
registradas = cliente.get_run(version.run_id).data.metrics
print(f'{URI} → versión {version.version}, run_id {version.run_id}, modelo {version.tags.get("modelo")}')
print(f"Registrado: RMSLE {registradas['rmsle_prueba']!r} · costo 1:3 {registradas['costo_1_3']!r}")

C:\Users\lroja\MSDS\Data Science\Pset3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


models:/favorita-perecederos-modelo@champion → versión 1, run_id 826c7f1e2fe7407d9e5873f43461868f, modelo chronos cuantil
Registrado: RMSLE 0.7827505330788056 · costo 1:3 184162.2633422793


In [2]:
modelo = mlflow.sklearn.load_model(URI)
ventas, meta = cargar_muestra()
_, X_prueba, _, y_prueba = dividir(construir_features(ventas), columnas_extra=SERIE + ['DATE'])
pedido = np.clip(modelo.predict(X_prueba), 0, None)
y = y_prueba.to_numpy(dtype=float)
reproducidas = {'rmsle_prueba': rmsle(y, pedido),
                'costo_1_3': 1 * np.maximum(pedido - y, 0).sum() + 3 * np.maximum(y - pedido, 0).sum()}
print(f'Huella de la muestra: {meta["sha256"]}')
for nombre, valor in reproducidas.items():
    exacta = np.isclose(valor, registradas[nombre], rtol=1e-12, atol=1e-12)
    print(f'{nombre}: reproducido {valor!r} · diferencia {abs(valor - registradas[nombre]):.2e} → '
          f'{"OK" if exacta else "revisa"}')

Huella de la muestra: 4c43165b8f2b72904e5163ee431b3d9dee18bec1c820ee2a4deb035538e899cc
rmsle_prueba: reproducido 0.7827505330788056 · diferencia 0.00e+00 → OK
costo_1_3: reproducido np.float64(184162.2633422793) · diferencia 0.00e+00 → OK
